# VisionVoice - Part 2: Baseline ResNet-LSTM

This notebook runs the full data pipeline, trains the baseline encoder-decoder captioning model, evaluates BLEU scores, and displays qualitative predictions. Run this after reviewing the EDA notebook.

# 2. Initialization


## 2.1 Environment Setup and Logging

In [ ]:
import os
import platform
import sys

print(f'Python : {sys.version}')
print(f'OS     : {platform.system()} {platform.release()}')
print(f'CWD    : {os.getcwd()}')

## 2.2 Libraries

Modern Deep Learning relies on a robust ecosystem of libraries. We utilize **PyTorch** for computational graphs and tensor operations, **NumPy** for numerical processing, and **Pandas** for structured data manipulation. We also implement a **Warning Filter** to suppress non-critical engine warnings, ensuring that our final notebook remains professional and readable for peer review or production handover.

In [ ]:
# Standard Python Utilities
import os, sys, gc, time, json, random, zipfile, warnings, shutil
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Iterable, Any
from collections import Counter

# Networking
import requests

# Data Processing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Image Processing
from PIL import Image, UnidentifiedImageError

# Visualization
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.models as models
import torchvision.transforms as transforms

# NLP & Progress
import nltk
from nltk.translate.bleu_score import corpus_bleu
from tqdm.notebook import tqdm

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Centralized Device Configuration
# This ensures we have a single source of truth for hardware acceleration
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Target Device: {DEVICE}")

## 2.3 Centralized Configuration (Settings)

A best practice in high-level Data Science is the **Singleton Configuration Pattern**. Instead of hardcoding hyperparameters like 'LEARNING_RATE' or 'BATCH_SIZE' inside functions, we centralize them in a `Settings` class. This facilitates **Hyperparameter Optimization (HPO)** and makes switching between 'train' and 'inference' modes seamless. We use `pathlib` for **Object-Oriented Filesystem Paths**, which is more robust than standard string manipulation.

In [ ]:
class Settings:
    """Centralized configuration class for the Image Captioning project.

    Contains all hyperparameters, environment settings, and dynamic path
    routing for both Kaggle and local VS Code environments.
    """

    # 1. Operational Toggles
    MODE: str = "train"   # "eda", "train", or "inference"
    DEBUG: bool = False
    DEBUG_SAMPLE_SIZE: int = 50 # Number of files to audit when DEBUG is True
    SHOW_PLOTS: bool = MODE == "eda"  # Keep training notebooks lightweight by default

    # 2. Environment Detection
    IS_KAGGLE: bool = Path("/kaggle/input").exists()

    # 3. Dynamic Path Management
    # Override these in Kaggle with environment variables if your dataset slug changes:
    # VIZWIZ_DATA_DIR, VIZWIZ_ANNOTATIONS_DIR, VIZWIZ_IMAGE_DIR
    if IS_KAGGLE:
        DATA_DIR = Path(os.getenv("VIZWIZ_DATA_DIR", "/kaggle/input/datasets/tuannm3823/vizwiz"))
        ANNOTATIONS_DIR = Path(os.getenv("VIZWIZ_ANNOTATIONS_DIR", str(DATA_DIR / "annotations" / "annotations")))
        TRAIN_IMG_DIR = Path(os.getenv("VIZWIZ_IMAGE_DIR", str(DATA_DIR / "val" / "val")))
        VAL_IMG_DIR = TRAIN_IMG_DIR
        WORK_DIR = Path("/kaggle/working")
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            DATA_DIR = Path(os.getenv("VIZWIZ_LOCAL_DATA_DIR", "/content/drive/MyDrive/Sem3_2026 Autumn/94691 Deep Learning/dl_assignments/dl_at3"))
        except ImportError:
            DATA_DIR = Path(os.getenv("VIZWIZ_LOCAL_DATA_DIR", "./data"))

        ANNOTATIONS_DIR = DATA_DIR / "annotations"
        TRAIN_IMG_DIR = DATA_DIR / "images"
        VAL_IMG_DIR = DATA_DIR / "images"
        WORK_DIR = DATA_DIR / "working"

    BASE: Path = DATA_DIR
    CACHE_DIR: Path = WORK_DIR / "cache"
    WORK_CACHE_DIR: Path = CACHE_DIR / "work"

    # /kaggle/input/models/tuannm3823/visionvoice-image-captioning/pytorch/attention-resnet-lstm/1/vision_voice_attention_best.pth
    # /kaggle/input/models/tuannm3823/visionvoice-image-captioning/pytorch/baseline-resnet-lstm/1/vision_voice_baseline_best.pth

    # 4. Remote Data URLs (For Section 3.1)
    URL_ANNOTATIONS: str = "https://vizwiz.cs.colorado.edu/VizWiz_final/annotations/val.json"
    URL_IMAGES: str = "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip"

    # 5. Data Processing & Pipeline Parameters
    SEED: int = 42
    VAL_SPLIT_SIZE: float = 0.2    # 20% of data used for validation
    VOCAB_MIN_FREQ: int = 2        # Minimum word occurrences to be indexed
    IMAGE_SIZE: int = 224          # ResNet standard input size
    NUM_WORKERS: int = 2           # DataLoader subprocesses

    # 6. Model Hyperparameters
    LEARNING_RATE: float = 1e-4
    BATCH_SIZE: int = 32
    EPOCHS: int = 10
    MAX_SEQ_LEN: int = 25          # Max words generated during inference
    EMBED_SIZE: int = 256          # Dimension of shared visual/textual space
    HIDDEN_SIZE: int = 512         # Number of units in LSTM hidden state
    NUM_LAYERS: int = 1            # Number of LSTM layers stacked
    MAX_GRAD_NORM: float = 5.0     # Clipping threshold for vanishing/exploding gradients

    # 7. Evaluation Parameters
    BLEU_EVAL_SAMPLES: int = 500   # Number of validation images to test for BLEU scoring

# Instantiate configuration object
CFG = Settings()

# Post-instantiation directory setup
CFG.CACHE_EXISTS = CFG.CACHE_DIR.exists()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODE         : {CFG.MODE}')
print(f'BASE         : {CFG.BASE}')
print(f'CACHE_EXISTS : {CFG.CACHE_EXISTS}')
print(f'CACHE_DIR    : {CFG.CACHE_DIR}')

## 2.4 Memory Optimization and Management

Deep Learning models are notorious for **RAM/VRAM** exhaustion. To mitigate **Out-of-Memory (OOM)** errors and ensure smooth execution, we implement two critical strategies:
1.  **Garbage Collection**: Manually triggering `gc.collect()` and `torch.cuda.empty_cache()` to proactively free up unused memory pointers.
2.  **Data Type Downcasting**: The `reduce_mem_usage` function iterates through a **Pandas DataFrame** and intelligently converts data types to their smallest possible representation (e.g., `float64` to `float32` or `int64` to `int8`). This significantly reduces the memory footprint of large datasets without losing critical precision.

In [ ]:
def clear_memory() -> None:
    """Forces garbage collection and clears VRAM cache to prevent OOM errors."""
    gc.collect()

    # Clear CUDA cache if using NVIDIA
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Clear MPS cache if using Apple Silicon
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

    if CFG.DEBUG:
        print("[Memory] Garbage collection forced and accelerator cache cleared.")


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """
    Iterates through all columns of a dataframe and downcasts data types
    to their smallest possible representation to reduce memory footprint.
    """
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        # Exclude object (string) and categorical types from numeric downcasting
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2

    if CFG.DEBUG:
        reduction = 100 * (start_mem - end_mem) / start_mem
        print(f"[Memory] DataFrame compressed from {start_mem:.2f}MB to {end_mem:.2f}MB ({reduction:.1f}% reduction).")

    return df

## 2.5 Seeding

Deep learning is inherently **stochastic**. Weights are initialized randomly, and data is shuffled. To ensure that our performance metrics are comparable across different runs and by different team members, we must enforce **Global Determinism**. We do this by locking the seeds for the Python `random` module, `numpy`, and `torch`. Crucially, we also disable the **cuDNN benchmark** and set it to deterministic mode to ensure identical results on GPU backends.

In [ ]:
def seed_everything(seed: int = CFG.SEED) -> None:
    """
    Locks all random number generators to a specific seed to ensure
    Global Determinism across independent experimental runs.
    """
    # 1. Python and Data structures
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch Core
    torch.manual_seed(seed)

    # 4. PyTorch CUDA (Hardware Acceleration)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Ensures determinism for multi-GPU setups

        # Enforce deterministic convolutions.
        # Note: This may slightly reduce training speed, but guarantees 100% reproducible results.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"Global seed successfully locked to: {seed}")

# Execute initialization
seed_everything(CFG.SEED)

# 3. Data Preparing


## 3.1 Data Loading and Preparation

In modern **MLOps pipelines**, manual data handling is a significant bottleneck and a primary source of environment drift. To ensure **reproducibility** and scalability, we must automate the acquisition and initial preparation of our raw assets. The **VizWiz dataset** consists of high-resolution images and complex JSON annotations, which are often too large to fit entirely into system memory (RAM) during the download phase.

Our approach employs **chunked streaming** to download large ZIP files efficiently in small, manageable pieces (typically 8KB to 1MB). This prevents **Out-of-Memory (OOM)** errors during download. After downloading, the function automates the **extraction** of image and annotation files into their designated directories. This structured approach ensures that downstream preprocessing steps have a consistent and predictable data source, whether running on Kaggle or a local Colab environment.

In [ ]:
def download_file(url: str, dest_dir: Path, filename: str = None) -> Path:
    """
    Downloads a file from a URL using chunked streaming to prevent OOM errors.

    Args:
        url (str): The source URL.
        dest_dir (Path): The local directory to save the file.
        filename (str, optional): Custom filename. Defaults to URL's trailing name.

    Returns:
        Path: The absolute path to the downloaded file.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    filename = filename or url.split("/")[-1]
    file_path = dest_dir / filename

    if file_path.exists():
        print(f"[Network] '{filename}' already exists in cache. Skipping download.")
        return file_path

    print(f"[Network] Initiating stream for '{filename}'...")
    try:
        # Stream=True prevents loading the entire file into RAM at once
        with requests.get(url, stream=True, timeout=15) as response:
            response.raise_for_status()
            with open(file_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
        print(f"[Network] Successfully downloaded: {file_path}")
    except requests.exceptions.RequestException as e:
        print(f"[Error] Failed to download {filename}: {e}")
        raise SystemExit("Critical data missing. Pipeline halted.")

    return file_path

def prepare_dataset() -> None:
    """
    Orchestrates the data pipeline: routing Kaggle paths or downloading/extracting
    the VizWiz dataset into the local Colab environment.
    """
    if CFG.IS_KAGGLE:
        print("[Pipeline] Kaggle environment detected. Utilizing read-only mounted datasets.")
        if not (CFG.ANNOTATIONS_DIR / "val.json").exists():
             print(f"[Warning] val.json not found in {CFG.ANNOTATIONS_DIR}.")
        return

    print("--- Initializing Local Data Pipeline ---")

    # 1. Acquire Annotations (Metadata)
    CFG.ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    download_file(
        url=CFG.URL_ANNOTATIONS,
        dest_dir=CFG.ANNOTATIONS_DIR,
        filename="val.json"
    )

    # 2. Acquire and Extract Images
    # We verify extraction success by checking if the directory exists and is populated
    if CFG.TRAIN_IMG_DIR.exists() and any(CFG.TRAIN_IMG_DIR.iterdir()):
        print(f"[Pipeline] Image payload verified at: {CFG.TRAIN_IMG_DIR}. Skipping extraction.")
    else:
        print(f"[Pipeline] Image directory empty. Beginning extraction protocol...")
        CFG.TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

        # Download the heavy zip file to the temporary cache
        zip_path = download_file(CFG.URL_IMAGES, CFG.WORK_CACHE_DIR)

        print(f"[Storage] Unpacking compressed images to {CFG.TRAIN_IMG_DIR}...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                # Extract to a temp cache folder to handle unexpected internal nested folders
                temp_extract_path = CFG.WORK_CACHE_DIR / "temp_extract"
                temp_extract_path.mkdir(exist_ok=True)
                zip_ref.extractall(temp_extract_path)

                # Flatten the directory structure if the zip contained a root 'val' folder
                extracted_items = list(temp_extract_path.iterdir())
                if len(extracted_items) == 1 and extracted_items[0].is_dir():
                    source_dir = extracted_items[0]
                else:
                    source_dir = temp_extract_path

                # Move files to final persistent directory
                for item in source_dir.iterdir():
                    shutil.move(str(item), str(CFG.TRAIN_IMG_DIR / item.name))

                # Housekeeping: Clean up temporary extraction folders
                shutil.rmtree(temp_extract_path)

            print("[Storage] Extraction and cleanup complete.")

        except zipfile.BadZipFile:
            print(f"[Error] Corrupted archive detected at {zip_path}.")
            zip_path.unlink() # Delete corrupted zip to force fresh download on next run
            raise SystemExit("Please re-run the cell to attempt download again.")

# Execute Pipeline
prepare_dataset()

## 3.2 Parsing and Cleaning Annotations

In Vision-Language (VL) tasks, the bridge between visual features and linguistic tokens is the **Metadata Annotation**. The VizWiz dataset follows a relational JSON structure (similar to COCO) where image metadata and textual captions are stored in separate lists linked by unique identifiers.

In [ ]:
print(f"--- Parsing Raw Annotations ---")
json_path = CFG.ANNOTATIONS_DIR / "val.json"

# 1. Load the Raw Data
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_df = pd.DataFrame(data['images'])
annotations_df = pd.DataFrame(data['annotations'])

# 2. Basic Text Cleanup (Fix legacy carriage returns that break tokenizers)
annotations_df['caption'] = annotations_df['caption'].str.replace('\r', '', regex=False)

# 3. Initial Merge (Attach file_names to captions for Visualization & EDA)
raw_corpus_df = pd.merge(
    annotations_df,
    images_df[['id', 'file_name']],
    left_on='image_id',
    right_on='id',
    how='inner'
)
# Rename the annotation's 'id' column to avoid confusion, drop the duplicate image 'id'
raw_corpus_df = raw_corpus_df.drop(columns=['id_y']).rename(columns={'id_x': 'annotation_id'})

print(f"Successfully loaded {len(raw_corpus_df)} raw annotations.\n")

**3.2.1 Loading Raw Data and Initial Visualization**

Before modifying the dataset, we must ingest the **raw JSON annotations** and inspect them. The VizWiz dataset is structured **relationally**, meaning the image metadata and the actual text captions are stored in separate JSON arrays.

In this step, we:
*   **Parse the JSON** and load it into **Pandas DataFrames**.
*   Perform a preliminary merge to link the `file_name` to its respective captions.
*   Utilize **Plotly** to visually inspect a random sample of the raw data, specifically highlighting captions that crowd-workers flagged as `[REJECTED]` (spam) or `[PRECANNED]` (quality issues).

In [ ]:
def show_raw_sample_with_plotly(df: pd.DataFrame, image_dir: Path, num_samples: int = 1) -> None:
    """Displays random images and color-codes noisy/spam captions for raw data inspection."""
    print(f"--- [Visualization] Sampling {num_samples} raw images ---")
    grouped = df.groupby('file_name')
    unique_files = list(grouped.groups.keys())

    safe_num_samples = min(num_samples, len(unique_files))
    samples = random.sample(unique_files, k=safe_num_samples)

    for file_name in samples:
        group = grouped.get_group(file_name)
        img_path = image_dir / file_name

        captions: List[str] = []
        colors: List[str] = []

        # Process Annotations and Apply Color Coding for Bad Data
        for _, row in group.iterrows():
            is_rejected = row.get('is_rejected', 0.0)
            is_precanned = row.get('is_precanned', 0.0)

            if is_rejected:
                captions.append(f"[REJECTED] {row['caption']}")
                colors.append('crimson')
            elif is_precanned:
                captions.append(f"[PRECANNED] {row['caption']}")
                colors.append('royalblue')
            else:
                captions.append(row['caption'])
                colors.append('black')

        # Setup Plotly Subplot
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.4, 0.6],
            specs=[[{"type": "image"}, {"type": "table"}]],
            horizontal_spacing=0.05
        )

        # Inject Image
        if img_path.exists():
            try:
                with Image.open(img_path) as img:
                    img_array = np.array(img.convert("RGB"))
                    fig.add_trace(go.Image(z=img_array), row=1, col=1)
            except Exception as e:
                print(f"[Warning] Could not render image {file_name}: {e}")

        # Inject Caption Table
        fig.add_trace(
            go.Table(
                header=dict(
                    values=[f"<b>Raw Captions for {file_name}</b>"],
                    align="left",
                    font=dict(color="white", size=14),
                    fill_color="#2C3E50"
                ),
                cells=dict(
                    values=[captions],
                    align="left",
                    font=dict(color=[colors], size=13),
                    height=40,
                    fill_color="#F8F9F9"
                )
            ),
            row=1, col=2
        )

        fig.update_layout(height=400, margin=dict(l=10, r=10, t=30, b=10), plot_bgcolor="white")
        fig.update_xaxes(visible=False, row=1, col=1)
        fig.update_yaxes(visible=False, row=1, col=1)
        fig.show()

# Execute Visualization
if CFG.SHOW_PLOTS:
    # Using a seed before sampling ensures we get the same random images every time we run the cell
    random.seed(CFG.SEED)
    show_raw_sample_with_plotly(raw_corpus_df, CFG.TRAIN_IMG_DIR, num_samples=3)

**3.2.2 Exploratory Data Analysis (Dataset Quality)**

Before we perform any destructive filtering on our dataset, we must understand the **distribution of noise**. The VizWiz dataset is highly authentic, meaning it contains unreadable images (flagged as `[PRECANNED]`) and conversational spam (flagged as `[REJECTED]`).

By visualizing the **annotation quality** and the **distribution of "clean" captions** per image, we ensure that our downstream filtering logic does not inadvertently create **"orphaned" images** (images with zero valid captions left), which would crash our PyTorch DataLoader.

In [ ]:
print("--- [EDA] Raw Dataset Quality Analysis ---")

# 1. Identify Quality Flags
# Safely handle JSON boolean/float parsing variations
is_p = (raw_corpus_df['is_precanned'] == True) | (raw_corpus_df['is_precanned'] == 1.0)
is_r = (raw_corpus_df['is_rejected'] == True) | (raw_corpus_df['is_rejected'] == 1.0)

# 2. Annotation-Level Intersection Analysis
both_count = (is_p & is_r).sum()
just_precanned = (is_p & ~is_r).sum()
just_rejected = (is_r & ~is_p).sum()
clean_count = (~is_p & ~is_r).sum()

print("\n[Annotation Breakdown]")
print(f"Total Annotations : {len(raw_corpus_df)}")
print(f"Clean/Descriptive : {clean_count}")
print(f"Precanned Only    : {just_precanned}")
print(f"Rejected Only     : {just_rejected}")
print(f"BOTH (P & R)      : {both_count}")

# 3. Image-Level "Good Caption" Distribution
raw_corpus_df['is_good'] = ~is_p & ~is_r
dist_counts = raw_corpus_df.groupby('file_name')['is_good'].sum().value_counts().sort_index()

if CFG.SHOW_PLOTS:
    # 4. Visualizations (Viridis Palette)
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "pie"}, {"type": "bar"}]],
        subplot_titles=("Raw Annotation Quality", "Descriptive Captions per Image")
    )

    viridis = px.colors.sequential.Viridis

    # Pie Chart
    fig.add_trace(go.Pie(
        labels=['Clean', 'Precanned', 'Rejected', 'Both'],
        values=[clean_count, just_precanned, just_rejected, both_count],
        hole=.4,
        marker_colors=[viridis[8], viridis[5], viridis[2], viridis[0]]
    ), row=1, col=1)

    # Bar Chart
    fig.add_trace(go.Bar(
        x=dist_counts.index,
        y=dist_counts.values,
        marker=dict(color=dist_counts.index, colorscale='Viridis'),
        text=dist_counts.values,
        textposition='auto'
    ), row=1, col=2)

    fig.update_layout(height=450, showlegend=True, plot_bgcolor="white", title_text="VizWiz Quality Breakdown")
    fig.update_xaxes(title_text="Valid Captions (0 to 5)", row=1, col=2, showgrid=True, gridcolor='lightgrey', dtick=1)
    fig.update_yaxes(title_text="Number of Images", row=1, col=2, showgrid=True, gridcolor='lightgrey')
    fig.show()

# Cleanup temporary EDA column
raw_corpus_df.drop(columns=['is_good'], inplace=True)

### Key EDA Insights

*   Exploratory Data Analysis revealed that while 85.5% of the VizWiz annotations were descriptive, 12% consisted of precanned 'quality issue' flags indicative of the dataset's in-the-wild nature.
*   Furthermore, filtering out this noise resulted in 208 completely orphaned images (0 valid captions) and a variable number of reference captions (1-4) for thousands of others.
*   This EDA directly informed our pipeline architecture, necessitating the removal of orphaned images to prevent DataLoader instability and requiring dynamic reference grouping for our BLEU evaluation.

**3.2.3 Data Cleaning and Leakage-Free Splitting**

Having identified the noise in our dataset, we now sterilize the corpus to optimize for our evaluation metric (**BLEU**). Because **BLEU** measures **n-gram overlap**, allowing standard conversational phrases or uniform "Quality issue" strings into the training set will artificially depress our model's scores.

In this step, we:

*   Filter: Drop all `[REJECTED]` and `[PRECANNED]` annotations.

*   Prune Orphans: Drop any images that no longer have valid text targets.

*   Split: Perform an **80/20 Train/Validation split** based on unique images, guaranteeing no **data leakage** between the sets.

In [ ]:
print("--- [Data Filtering] ---")
original_anno_count = len(raw_corpus_df)

# 1. Silently drop Spam and Precanned
clean_corpus_df = raw_corpus_df[
    (raw_corpus_df['is_rejected'] != True) &
    (raw_corpus_df['is_rejected'] != 1.0) &
    (raw_corpus_df['is_precanned'] != True) &
    (raw_corpus_df['is_precanned'] != 1.0)
].reset_index(drop=True)

spam_removed = original_anno_count - len(clean_corpus_df)
print(f"Dropped {spam_removed} noisy annotations.")

# 2. Identify and drop orphaned images (images with 0 valid captions)
# By grouping by file_name, we ensure we only keep images that still exist in clean_corpus_df
valid_images = clean_corpus_df['file_name'].unique()
print(f"Final valid corpus size: {len(clean_corpus_df)} captions across {len(valid_images)} images.\n")

print("--- [Splitting] ---")
# 3. Perform a leakage-free split
train_imgs, val_imgs = train_test_split(
    valid_images,
    test_size=CFG.VAL_SPLIT_SIZE,
    random_state=CFG.SEED
)

train_df = clean_corpus_df[clean_corpus_df['file_name'].isin(train_imgs)].reset_index(drop=True)
val_df = clean_corpus_df[clean_corpus_df['file_name'].isin(val_imgs)].reset_index(drop=True)

# Compress datatypes to save memory
train_df = reduce_mem_usage(train_df)
val_df = reduce_mem_usage(val_df)

print(f"Train Split : {len(train_imgs)} images | {len(train_df)} total captions")
print(f"Val Split   : {len(val_imgs)} images | {len(val_df)} total captions\n")

# 4. Handle Debug Truncation
if CFG.DEBUG:
    print("[Data Prep] DEBUG MODE: Truncating DataFrames for rapid iteration.")
    train_df = train_df.head(100)
    val_df = val_df.head(100)

## 3.3 Building the Vocabulary

In Computer Vision and Natural Language Processing (CV-NLP) tasks like Image Captioning, the model cannot directly interpret raw strings. We must map every unique word to a specific integer representation, a process known as **Numericalization**.

To manage this, we implement a custom `Vocabulary` class. This utility serves several critical MLOps and Data Science purposes:
1.  **Special Token Handling**: We reserve specific IDs for **Special Tokens** such as `<PAD>` (padding sequences to equal length), `<START>`/`<END>` (marking sequence boundaries), and `<UNK>` (Unknown tokens for words not seen during training).
2.  **Frequency Thresholding**: By using a `freq_threshold`, we filter out rare words or typos. This reduces the **Dimensionality** of our embedding layer and prevents the model from overfitting on noise.
3.  **Bidirectional Mapping**: We maintain two dictionaries, 'stoi' (**String-to-Index**) for encoding and 'itos' (**Index-to-String**) for decoding model predictions back into human-readable text.

In [ ]:
class Vocabulary:
    """
    Handles the bidirectional mapping between string tokens and integer IDs.
    """
    def __init__(self, freq_threshold: int = CFG.VOCAB_MIN_FREQ) -> None:
        self.freq_threshold = freq_threshold

        # Initialize dictionaries with reserved MLOps special tokens
        self.itos: Dict[int, str] = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.stoi: Dict[str, int] = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.idx: int = 4

    def __len__(self) -> int:
        """Returns the total number of unique words in the vocabulary."""
        return len(self.itos)

    def tokenize(self, text: str) -> List[str]:
        """Converts raw text to lowercase and isolates punctuation."""
        text = str(text).lower()
        # Ensure punctuation is treated as separate tokens to improve context learning
        for punc in [".", ",", "!", "?", "\"", "'", "(", ")"]:
            text = text.replace(punc, f" {punc} ")
        return text.split()

    def build_vocabulary(self, sentence_list: List[str]) -> None:
        """Populates the stoi and itos dictionaries based on token frequencies."""
        print(f"--- [Vocabulary] Building index with threshold >= {self.freq_threshold} ---")
        frequencies = Counter()

        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1

        # Only add words that meet the frequency requirement to prevent overfitting on typos
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

        print(f"[Vocabulary] Processed {len(sentence_list)} sentences. Indexed {len(self.itos)} unique tokens.")

    def numericalize(self, text: str) -> List[int]:
        """Converts a raw string into a list of integer tensor indices."""
        tokenized_text = self.tokenize(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]

In [ ]:
# 1. Extract all clean training captions
# We ONLY build the vocabulary on the training set to prevent data leakage from the val set
train_captions = train_df['caption'].tolist()

# 2. Instantiate and build the Vocabulary object
vocab = Vocabulary(freq_threshold=CFG.VOCAB_MIN_FREQ)
vocab.build_vocabulary(train_captions)

# 3. Validation Check
if CFG.DEBUG or CFG.MODE == "train":
    # Test the numericalization pipeline on the first available caption
    sample_text = train_captions[0]
    print(f"\n[Validation] Source Text : {sample_text}")
    print(f"[Validation] Tokenized   : {vocab.tokenize(sample_text)}")
    print(f"[Validation] Encoded IDs : {vocab.numericalize(sample_text)}")

In [ ]:
def analyze_nlp_distributions(captions: List[str], vocab: Vocabulary) -> None:
    """
    Analyzes the distribution of caption lengths to inform the MAX_SEQ_LEN hyperparameter,
    and visualizes the most frequent words in the dataset.
    """
    print("--- [EDA] NLP Token Distribution Analysis ---")

    # 1. Calculate Sequence Lengths
    # We add +2 to account for the <START> and <END> tokens we will inject later
    seq_lengths = [len(vocab.tokenize(cap)) + 2 for cap in captions]

    # 2. Calculate Word Frequencies
    all_words = []
    for cap in captions:
        all_words.extend(vocab.tokenize(cap))
    word_counts = pd.Series(all_words).value_counts()

    # 3. Compute the 95th Percentile
    percentile_95 = np.percentile(seq_lengths, 95)

    # 4. Visualizations (Plotly)
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "histogram"}, {"type": "bar"}]],
        subplot_titles=("Caption Length Distribution (Includes Start/End)", "Top 20 Most Frequent Words")
    )

    # Histogram of Sequence Lengths
    fig.add_trace(go.Histogram(
        x=seq_lengths,
        nbinsx=40,
        marker_color='#9B59B6',
        name="Length Count"
    ), row=1, col=1)

    # Add a visual line for the 95th percentile
    fig.add_vline(
        x=percentile_95, line_dash="dash", line_color="red", line_width=2,
        annotation_text=f" 95th Percentile: {int(percentile_95)} words ",
        annotation_position="top right",
        row=1, col=1
    )

    # Bar Chart of Top 20 Words
    top_20 = word_counts.head(20)
    fig.add_trace(go.Bar(
        x=top_20.index,
        y=top_20.values,
        marker=dict(color=top_20.values, colorscale='Viridis'),
        text=top_20.values,
        textposition='outside',
        name="Frequency"
    ), row=1, col=2)

    fig.update_layout(height=450, showlegend=False, plot_bgcolor="white")
    fig.update_xaxes(title_text="Number of Tokens", row=1, col=1, showgrid=True, gridcolor='lightgrey')
    fig.update_yaxes(title_text="Count", row=1, col=1, showgrid=True, gridcolor='lightgrey')
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.show()

    # Console Summary
    print(f"\n[NLP Summary]")
    print(f"Shortest Caption : {min(seq_lengths)} tokens")
    print(f"Longest Caption  : {max(seq_lengths)} tokens")
    print(f"Average Caption  : {int(np.mean(seq_lengths))} tokens")
    print(f"95% Threshold    : {int(percentile_95)} tokens")
    print(f"Actionable Insight: Set CFG.MAX_SEQ_LEN to ~{int(percentile_95)} to capture the vast majority of data without wasting VRAM.")

# Execute the EDA on the clean training captions
if CFG.SHOW_PLOTS:
    analyze_nlp_distributions(train_captions, vocab)

### Key NLP Insights:

*   **Insight 1: Optimizing the Sequence Length (`MAX_SEQ_LEN`)**

    Analysis of the tokenized sequence lengths revealed a heavy right-skewed distribution. While the average caption length was only 15 tokens, extreme outliers extended up to 175 tokens. To optimize **VRAM** utilization and training efficiency, we set the model's `MAX_SEQ_LEN` hyperparameter to 25. This covers the 95th percentile of all data (24 tokens), ensuring the model learns complete sentences while avoiding the immense computational waste of padding 95% of batches to accommodate single 175-word outliers.

*   **Insight 2: Vocabulary Composition & Domain Bias**

    Visualizing the most frequent tokens confirmed the expected dominance of English stop words ('a', 'on', 'the'). However, the emergence of specific descriptive nouns and adjectives in the top 20—such as 'white', 'black', 'computer', 'screen', 'table', and 'bottle'—highlights a distinct **domain bias** in the VizWiz dataset toward indoor, desktop, and everyday household environments. This informs our qualitative evaluation expectations, as the model will likely perform best on these heavily represented indoor object classes.

## 3.4 Creating the Custom PyTorch Dataset and DataLoader
In **multimodal deep learning**, a **robust data pipeline** is essential to prevent the CPU from bottlenecking the GPU. We implement this pipeline in three parts:

1. **Image Transformations**: We apply a standard **ResNet preprocessing pipeline**. Images are resized to 224x224, converted to tensors, and normalized using ImageNet's mathematical mean and standard deviation.

2. The **VizWizDataset** (**Lazy Loading**): Inheriting from `torch.utils.data.Dataset`, this class loads images from disk only when requested by the batch, preventing **Out-Of-Memory (OOM) RAM crashes**. It also numericalizes the text and truncates any extreme outliers based on the `CFG.MAX_SEQ_LEN` we established during EDA.

3. The **CapsCollate Function**: PyTorch requires all tensors in a batch to have identical dimensions. Because captions have variable lengths, we cannot use the default collate function. `CapsCollate` dynamically pads all sequences in a specific batch with the `<PAD>` token to match the length of the longest sequence in that batch.

In [ ]:
class VizWizDataset(Dataset):
    """
    Custom PyTorch Dataset for loading images and numericalized captions.
    """
    def __init__(self, df: pd.DataFrame, img_dir: Path, vocab: Vocabulary, transform: transforms.Compose, max_seq_len: int):
        self.df = df
        self.img_dir = img_dir
        self.vocab = vocab
        self.transform = transform
        self.max_seq_len = max_seq_len

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        caption = self.df.iloc[index]['caption']
        img_filename = self.df.iloc[index]['file_name']
        img_path = self.img_dir / img_filename

        # 1. Load and Transform Image
        # .convert("RGB") is crucial to drop Alpha channels from PNGs or convert Grayscale
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        # 2. Numericalize Text
        numericalized_caption = [self.vocab.stoi["<START>"]]
        numericalized_caption.extend(self.vocab.numericalize(caption))
        numericalized_caption.append(self.vocab.stoi["<END>"])

        # 3. Truncate Extreme Outliers
        # If the caption exceeds our EDA-derived MAX_SEQ_LEN, we cut it short but preserve the <END> token
        if len(numericalized_caption) > self.max_seq_len:
            numericalized_caption = numericalized_caption[:self.max_seq_len - 1]
            numericalized_caption.append(self.vocab.stoi["<END>"])

        return img, torch.tensor(numericalized_caption)


class CapsCollate:
    """
    Custom collate function to handle variable-length text sequences within a batch.
    """
    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch: List[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
        # Extract images and stack them into a 4D tensor: [batch_size, channels, height, width]
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)

        # Extract textual targets
        targets = [item[1] for item in batch]

        # Dynamically pad sequences to the longest sequence *in this specific batch*
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets

In [ ]:
print("--- [Data Loader] Initializing PyTorch Pipelines ---")

# 1. Define Standard ImageNet Transformations
transform_pipeline = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Get the Padding Index from the Vocabulary
pad_idx = vocab.stoi["<PAD>"]

# 3. Instantiate Datasets
train_dataset = VizWizDataset(
    df=train_df, img_dir=CFG.TRAIN_IMG_DIR, vocab=vocab,
    transform=transform_pipeline, max_seq_len=CFG.MAX_SEQ_LEN
)

val_dataset = VizWizDataset(
    df=val_df, img_dir=CFG.VAL_IMG_DIR, vocab=vocab,
    transform=transform_pipeline, max_seq_len=CFG.MAX_SEQ_LEN
)

# 4. Instantiate DataLoaders
# pin_memory=True speeds up the transfer of data from CPU RAM to GPU VRAM
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,           # Shuffle training data to break correlation between consecutive batches
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,          # Never shuffle validation data; order doesn't matter for evaluation
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

print(f"Train Loader : {len(train_loader)} batches (Batch Size: {CFG.BATCH_SIZE})")
print(f"Val Loader   : {len(val_loader)} batches (Batch Size: {CFG.BATCH_SIZE})")

# 5. Pipeline Sanity Check (Inspect the first batch)
if CFG.DEBUG or CFG.MODE == "train":
    print("\n--- [Data Loader] First Batch Sanity Check ---")
    dataiter = iter(train_loader)
    images, captions = next(dataiter)

    print(f"Images Tensor Shape  : {images.shape}  -> [Batch, Channels, Height, Width]")
    print(f"Captions Tensor Shape: {captions.shape}   -> [Batch, Sequence_Length]")

# 4. Defining the Architecture (Model 1)

As per the assignment requirements, we will implement an **Encoder-Decoder** architecture. The **Encoder** will act as the "eyes," extracting high-level features from the images, while the **Decoder** will act as the "voice," generating human-readable sentences token by token.

## 4.1 The Encoder (Feature Extractor)

For the **Encoder**, we utilize **Transfer Learning** by employing a **pre-trained ResNet-50 model**. This model has already been trained on millions of images in the **ImageNet dataset**, meaning it already understands how to detect edges, textures, and complex objects.  Our adaptation strategy:

*   **Layer Removal**: we remove the final **fully-connected (classification) layer** of the ResNet-50. We don't need to classify the image into one of 1,000 categories; we only need the raw **feature vector**.
*   **Backbone Freezing**: We freeze the weights of the ResNet layers to preserve the pre-trained knowledge and reduce computational cost during training.
*   **Linear Projection**: We add a new **linear layer** to project the 2048-dimensional ResNet features into the `EMBED_SIZE` (256) used by our **Decoder**.

In [ ]:
class EncoderCNN(nn.Module):
    """
    Utilizes a pre-trained ResNet-50 to extract visual features and
    projects them into a shared embedding space.
    """
    def __init__(self, embed_size: int):
        super(EncoderCNN, self).__init__()
        # Load pre-trained ResNet-50
        resnet = models.resnet50(pretrained=True)

        # Freeze parameters so they are not updated during training
        for param in resnet.parameters():
            param.requires_grad = False

        # Remove the final classification layer (fc)
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)

        # Linear layer to project features to embed_size
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)
        self.batch_norm = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # Extract features: [batch_size, 2048, 1, 1]
        with torch.no_grad():
            features = self.resnet(images)

        # Flatten: [batch_size, 2048]
        features = features.view(features.size(0), -1)

        # Project and Normalize: [batch_size, embed_size]
        return self.batch_norm(self.linear(features))

## 4.2 The Decoder (Sequence Generator)

The **Decoder** is an **LSTM (Long Short-Term Memory)** network. It receives the visual features from the **Encoder** and learns to predict the next word in the sequence based on the image context and all previous words.

**Key Implementation Details:**
*   **Embedding Layer**: Converts our integer word IDs into dense vectors.
*   **LSTM Cell**: Processes the sequence, maintaining a "hidden state" that represents the memory of the sentence generated so far.
*   **Linear Classifier**: Projects the LSTM output back to the size of our **Vocabulary** (6,439) to determine the probability of the next word.

In [ ]:
class DecoderRNN(nn.Module):
    """
    An LSTM-based generator that produces captions conditioned
    on visual features extracted by the Encoder.
    """
    def __init__(self, embed_size: int, hidden_size: int, vocab_size: int, num_layers: int):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        # Remove the <END> token for training (Teacher Forcing)
        # captions input: [<START>, w1, w2, ..., wN-1]
        embeddings = self.embed(captions[:, :-1])

        # Concatenate image features as the very first 'word' in the sequence
        # combined: [batch_size, sequence_length, embed_size]
        combined = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        # LSTM pass
        hiddens, _ = self.lstm(combined)

        # Project to vocabulary logits
        outputs = self.linear(hiddens)

        # Discard the very first prediction (which was based on the image feature alone)
        # to align with targets: [w1, w2, ..., wN-1, <END>]
        return outputs[:, 1:, :]

## 4.3 Model Assembly and Configuration
We wrap the sub-modules into an `ImageCaptioningModel` class. This high-level wrapper simplifies the training loop by abstracting the interaction between the **CNN** and the **LSTM**. We also perform a **Parameter Audit** to ensure our **ResNet** backbone is correctly frozen, which is a critical step for verifying that our training focus remains on the newly added projection and language layers.

In [ ]:
class ImageCaptioningModel(nn.Module):
    """
    Unified interface for the VisionVoice architecture.
    Coordinates the flow between visual extraction and text generation.
    """
    def __init__(self, encoder: nn.Module, decoder: nn.Module):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        # Step 1: Extract visual features
        features = self.encoder(images)
        # Step 2: Generate sequence predictions
        outputs = self.decoder(features, captions)
        return outputs

# --- Model Instantiation ---

# Initialize the sub-modules using our centralized CFG parameters
encoder_net = EncoderCNN(embed_size=CFG.EMBED_SIZE)
decoder_net = DecoderRNN(
    embed_size=CFG.EMBED_SIZE,
    hidden_size=CFG.HIDDEN_SIZE,
    vocab_size=len(vocab),
    num_layers=CFG.NUM_LAYERS
)

# Combine into the final model and transfer to the target DEVICE (CUDA/MPS/CPU)
cap_model = ImageCaptioningModel(encoder_net, decoder_net).to(DEVICE)

# --- Parameter Audit ---
if CFG.MODE == "train" or CFG.DEBUG:
    total_params = sum(p.numel() for p in cap_model.parameters())
    trainable_params = sum(p.numel() for p in cap_model.parameters() if p.requires_grad)

    print(f"--- [Architecture Audit] ---")
    print(f"Total Parameters     : {total_params:,}")
    print(f"Trainable Parameters : {trainable_params:,}")
    print(f"Frozen Parameters    : {total_params - trainable_params:,}")
    print(f"Target Device        : {DEVICE}")

    # Verify that the majority of parameters (the ResNet backbone) are frozen
    freeze_ratio = (1 - (trainable_params / total_params)) * 100
    print(f"Backbone Freeze Ratio: {freeze_ratio:.1f}%")

## 4.4 Defining the Loss and Optimizer
For **Image Captioning**, we treat the problem as a **classification task** at each time step. We use **Cross-Entropy Loss**, but with a critical modification: we must ignore the `<PAD>` tokens. If we don't, the model will "cheat" by learning to predict long strings of padding, which artificially inflates its performance without generating meaningful language. For the optimizer, we use **Adam** with our configured learning rate. We strictly filter the parameters passed to the optimizer to include only those where `requires_grad=True`, ensuring we don't waste computation on the **frozen CNN layers**.

In [ ]:
# 1. Define Loss Function
# ignore_index ensures the model isn't penalized/rewarded for padding tokens
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])

# 2. Define Optimizer
# We only pass the trainable parameters (Projection + LSTM + Linear layers)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, cap_model.parameters()),
    lr=CFG.LEARNING_RATE
)

print(f"--- [Optimization] ---")
print(f"Loss Function : {criterion}")
print(f"Optimizer     : Adam (LR={CFG.LEARNING_RATE})")

## 4.5 The Training Loop
To train the model, we use a technique called **Teacher Forcing**. During this process, we feed the ground-truth captions into the decoder so it learns to predict the next word in a sequence based on the "perfect" previous words.  To implement this correctly, we perform a target shift:

Input: Images and captions excluding the final `<END>` token.
Targets: Captions excluding the initial `<START>` token.

This forces the **LSTM** to predict the actual "next" word at every step. We also implement **Gradient Clipping** to prevent the "Exploding Gradient" problem, which is common in recurrent architectures like LSTMs.

In [ ]:
def train_with_early_stopping(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    epochs: int,
    patience: int = 3
) -> Tuple[List[float], List[float]]:
    """
    Trains the model with a validation pass and Early Stopping logic.
    Saves only the best model weights based on Validation Loss.
    """
    train_losses = []
    val_losses = []

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_model_path = CFG.WORK_DIR / "vision_voice_baseline_best.pth"

    print(f"--- [Training] Starting Baseline with Early Stopping (Patience: {patience}) ---")

    for epoch in range(epochs):
        # --- 1. Training Phase ---
        model.train()
        running_train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

        for images, captions in pbar:
            images, captions = images.to(device), captions.to(device)
            optimizer.zero_grad()
            outputs = model(images, captions)
            targets = captions[:, 1:]
            loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.MAX_GRAD_NORM)
            optimizer.step()
            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- 2. Validation Phase ---
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                outputs = model(images, captions)
                targets = captions[:, 1:]
                loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
                running_val_loss += loss.item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # --- 3. Early Stopping & Best Model Saving ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # Save the best model only
            torch.save(model.state_dict(), best_model_path)
            print(f"   --> New Best Val Loss! Model saved to {best_model_path.name}")
        else:
            epochs_without_improvement += 1
            print(f"   --> No improvement for {epochs_without_improvement} epoch(s).")

        if epochs_without_improvement >= patience:
            print(f"\n[Early Stopping] Validation loss hasn't improved in {patience} epochs. Hitting the brakes.")
            break

    return train_losses, val_losses

# --- Execute Training ---
if CFG.MODE == "train":
    clear_memory()
    train_log, val_log = train_with_early_stopping(
        model=cap_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epochs=CFG.EPOCHS,
        patience=3 # Stop if 3 epochs pass with no improvement
    )

## 4.6 Loss Visualization
After training, it is vital to visualize the loss curve. A healthy model should show a consistent downward trend. If the curve plateaus too early, we may need a higher learning rate; if it fluctuates wildly, we may need a lower one.

In [ ]:
if CFG.SHOW_PLOTS:
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=train_log, mode='lines', name='Train Loss', line=dict(color='#2ECC71')))
    fig.add_trace(go.Scatter(y=val_log, mode='lines', name='Val Loss', line=dict(color='#E74C3C')))
    fig.update_layout(title="Training vs Validation Loss", xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()

    fig.update_layout(
        title="VisionVoice Baseline Training Loss",
        xaxis_title="Epoch",
        yaxis_title="Cross-Entropy Loss",
        plot_bgcolor="white",
        height=450
    )
    fig.update_xaxes(showgrid=True, gridcolor='lightgrey')
    fig.update_yaxes(showgrid=True, gridcolor='lightgrey')
    fig.show()

# 5. Evaluation (Model 1)

In this phase, we move beyond the training loop to assess how well our model has learned to map visual features to linguistic sequences. Evaluation in image captioning is unique because there isn't one "correct" answer; a single image can be described in many valid ways. We will employ both **quantitative metrics** and **qualitative visual inspection** to judge the performance of our 'model'.

## 5.1 The Inference Engine

During the training phase, we utilized a technique called **Teacher Forcing**, where the ground-truth caption is fed into the decoder to predict the next token. However, during **Inference** (production), we do not have the ground truth.

The model must operate **autoregressively**:
1. It receives an image feature vector from the **Encoder**.
2. It is primed with a special `'<START>'` token.
3. It predicts the probability distribution of the next word.
4. We use **Greedy Search** to select the word with the highest probability.
5. This predicted word is fed back into the model as the input for the next time step.
6. This cycle repeats until the model generates an `'<END>'` token or reaches the 'max_len'.

**Task: Define a function 'generate_caption' that performs inference on a single image.**



In [ ]:
def generate_caption(model: nn.Module, image: torch.Tensor, vocab: Vocabulary) -> List[str]:
    """
    Generates a caption for a single image using greedy search.
    """
    model.eval()
    result_caption = []

    with torch.no_grad():
        # Add batch dimension and move to device
        image = image.unsqueeze(0).to(DEVICE)
        
        # 1. Get visual features from Encoder
        features = model.encoder(image)
        states = None # Initialize LSTM hidden/cell states
        
        # 2. Start with the <START> token
        current_word_idx = torch.tensor([vocab.stoi["<START>"]]).to(DEVICE)
        
        for _ in range(CFG.MAX_SEQ_LEN):
            # 3. Predict the next word
            embeddings = model.decoder.embed(current_word_idx).unsqueeze(0)
            hiddens, states = model.decoder.lstm(embeddings, states)
            outputs = model.decoder.linear(hiddens.squeeze(0))
            
            # 4. Greedy Search: Pick the word with the highest probability
            predicted_idx = outputs.argmax(1)
            predicted_word = vocab.itos[predicted_idx.item()]
            
            if predicted_word == "<END>":
                break
                
            result_caption.append(predicted_word)
            current_word_idx = predicted_idx

    return result_caption

## 5.2 Calculating BLEU Scores

To evaluate the model's accuracy at scale, we use the **BLEU (Bilingual Evaluation Understudy)** score. This is a standard NLP metric that calculates the **n-gram overlap** between the model's generated 'hypothesis' and one or more human-written 'references'.

*   **BLEU-1 (Unigrams):** Measures how many individual words match. It focuses on vocabulary accuracy.
*   **BLEU-4 (4-grams):** Measures how many 4-word sequences match. It serves as a proxy for grammatical fluency and coherence.

In this step, we iterate through the validation set, generate a hypothesis for each image, and compare it against the five reference captions provided by humans.

**Task: Iterate through a validation sample, generate captions, and calculate the corpus-level BLEU scores using 'nltk'.**

In [ ]:
def calculate_corpus_bleu(model: nn.Module, df: pd.DataFrame, vocab: Vocabulary):
    """
    Evaluates the model across the validation set and calculates BLEU scores.
    """
    print(f"--- [Evaluation] Calculating BLEU scores on {CFG.BLEU_EVAL_SAMPLES} samples ---")
    
    # Load the best weights saved during training
    best_path = CFG.WORK_DIR / "vision_voice_baseline_best.pth"
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    references_corpus = []
    hypotheses_corpus = []
    
    # Group by image to get all available clean references
    grouped = df.groupby('file_name')
    sample_files = random.sample(list(grouped.groups.keys()), min(CFG.BLEU_EVAL_SAMPLES, len(grouped)))

    for file_name in tqdm(sample_files, desc="Generating Captions"):
        img_path = CFG.VAL_IMG_DIR / file_name
        
        # 1. Prepare References
        group = grouped.get_group(file_name)
        refs = [vocab.tokenize(row['caption']) for _, row in group.iterrows()]
        references_corpus.append(refs)
        
        # 2. Generate Hypothesis
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img)
        hypothesis = generate_caption(model, img_tensor, vocab)
        hypotheses_corpus.append(hypothesis)

    # 3. Calculate Scores
    b1 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(1.0, 0, 0, 0))
    b2 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.5, 0.5, 0, 0))
    b3 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.33, 0.33, 0.33, 0))
    b4 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.25, 0.25, 0.25, 0.25))

    print(f"\n[Final Results]")
    print(f"BLEU-1: {b1:.4f} | BLEU-2: {b2:.4f} | BLEU-3: {b3:.4f} | BLEU-4: {b4:.4f}")
    return b1, b2, b3, b4

# Run evaluation
if CFG.MODE in {"train", "inference"}:
    b_results = calculate_corpus_bleu(cap_model, val_df, vocab)

## 5.3 Visual Inspection

Metrics like **BLEU** are strictly mathematical and often fail to capture the semantic nuance of language. For example, a model might describe a "dog" as a "canine"; the BLEU score would penalize this as a mismatch, even though it is factually correct.

**Visual Inspection** allows the researcher to see if the model's errors are "logical" (e.g., misidentifying a specific breed of dog) or "hallucinatory" (e.g., seeing a car where there is none).

**Task: Plot random validation images alongside our model's generated caption and the actual human references.**

In [ ]:
def inspect_model_visually(model, df, vocab, num_samples=3):
    """
    Displays random validation images with predicted vs. reference captions.
    """
    model.eval()
    
    # Group validation data to access all references
    grouped = df.groupby('file_name')
    sample_files = random.sample(list(grouped.groups.keys()), num_samples)

    for file_name in sample_files:
        img_path = CFG.VAL_IMG_DIR / file_name
        
        # 1. Generate Prediction
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform_pipeline(img)
        predicted_words = generate_caption(model, img_tensor, vocab)
        predicted_sentence = " ".join(predicted_words).capitalize() + "."

        # 2. Setup Visualization
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.show()

        print(f"--- [Qualitative Check] {file_name} ---")
        # Bold and Color formatting for professional output
        print(f"\033[1mPredicted: \033[92m{predicted_sentence}\033[0m") 
        
        print("\nGround Truth References:")
        group = grouped.get_group(file_name)
        for i, (_, row) in enumerate(group.iterrows(), 1):
            print(f"  {i}. {row['caption']}")
        print("="*60 + "\n")

# Execute visual check
if CFG.SHOW_PLOTS:
    inspect_model_visually(cap_model, val_df, vocab, num_samples=3)